<a href="https://colab.research.google.com/github/239-blip/mssp607/blob/main/Assignments/assignment1_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

read

In [1]:
!git clone https://github.com/239-blip/mssp607
%cd mssp607
import pandas as pd
ap=pd.read_excel("data/data_academic_performance.xlsx",sheet_name="SABER11_SABERPRO")

Cloning into 'mssp607'...
remote: Enumerating objects: 94, done.
remote: Counting objects: 100% (94/94), done.
remote: Compressing objects: 100% (83/83), done.
remote: Total 94 (delta 33), reused 5 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (94/94), 3.16 MiB | 6.31 MiB/s, done.
Resolving deltas: 100% (33/33), done.
/content/mssp607


In [2]:
#quality check
quality_summary = pd.DataFrame({
    "dtype": ap.dtypes.astype(str),
    "missing": ap.isna().sum(),
    "missing_pct": ap.isna().mean().mul(100).round(2),
    "unique_values": ap.nunique()
})
quality_summary

,dtype,missing,missing_pct,unique_values
COD_S11,object,0,0.0,12411
GENDER,object,0,0.0,2
EDU_FATHER,object,0,0.0,12
EDU_MOTHER,object,0,0.0,12
OCC_FATHER,object,0,0.0,12
OCC_MOTHER,object,0,0.0,12
STRATUM,object,0,0.0,7
SISBEN,object,0,0.0,6
PEOPLE_HOUSE,object,0,0.0,13
Unnamed: 9,float64,12411,100.0,0


Does parental education influence outcomes, and if so, how much?

In [3]:
display(ap['EDU_FATHER'].unique())
display(ap['EDU_MOTHER'].unique())

array(['Incomplete Professional Education', 'Complete Secundary',
       'Not sure', 'Complete professional education',
       'Incomplete Secundary', 'Incomplete technical or technological',
       'Complete technique or technology', 'Incomplete primary ',
       'Complete primary ', 'Postgraduate education', 0, 'Ninguno'],
      dtype=object)

array(['Complete technique or technology',
       'Complete professional education', 'Not sure',
       'Complete Secundary', 'Incomplete technical or technological',
       'Incomplete primary ', 'Incomplete Secundary',
       'Incomplete Professional Education', 'Postgraduate education',
       'Complete primary ', 0, 'Ninguno'], dtype=object)

In [4]:
#clean up
df1=ap[~ap['EDU_FATHER'].isin([0,'0','Not sure']) & ~ap['EDU_MOTHER'].isin([0,'0','Not sure'])].copy()
df1['EDU_FATHER']=df1['EDU_FATHER'].str.strip()
df1['EDU_MOTHER']=df1['EDU_MOTHER'].str.strip()

In [5]:
#give catagories values, bed=basic edu, the=tech higher edu, phe=professional higher edu
bed_map={
    "Ninguno": 0,
    "Incomplete primary": 1,
    "Complete primary": 2,
    "Incomplete Secundary": 3,
    "Complete Secundary": 4,
    "Incomplete technical or technological": 4,
    "Complete technique or technology": 4,
    "Incomplete Professional Education": 4,
    "Complete professional education": 4,
    "Postgraduate education": 4
}
the_map={
    "Ninguno": 0,
    "Incomplete primary": 0,
    "Complete primary": 0,
    "Incomplete Secundary": 0,
    "Complete Secundary": 0,
    "Incomplete technical or technological": 1,
    "Complete technique or technology": 2,
    "Incomplete Professional Education": 0,
    "Complete professional education": 0,
    "Postgraduate education": 0
}
phe_map={
    "Ninguno": 0,
    "Incomplete primary": 0,
    "Complete primary": 0,
    "Incomplete Secundary": 0,
    "Complete Secundary": 0,
    "Incomplete technical or technological": 0,
    "Complete technique or technology": 0,
    "Incomplete Professional Education": 1,
    "Complete professional education": 2,
    "Postgraduate education": 3
}

In [6]:
#create new column
df1['EDU_BC_PARENTS']=(df1['EDU_FATHER'].map(bed_map)+df1['EDU_MOTHER'].map(bed_map))
df1['EDU_TH_PARENTS']=(df1['EDU_FATHER'].map(the_map)+df1['EDU_MOTHER'].map(the_map))
df1['EDU_PH_PARENTS']=(df1['EDU_FATHER'].map(phe_map)+df1['EDU_MOTHER'].map(phe_map))

#check
df1[
    [
        "EDU_FATHER",
        "EDU_MOTHER",
        "EDU_BC_PARENTS",
        "EDU_TH_PARENTS",
        "EDU_PH_PARENTS"
    ]
].head(20)

,EDU_FATHER,EDU_MOTHER,EDU_BC_PARENTS,EDU_TH_PARENTS,EDU_PH_PARENTS
0,Incomplete Professional Education,Complete technique or technology,8,2,1
1,Complete Secundary,Complete professional education,8,0,2
4,Complete professional education,Complete professional education,8,0,4
5,Complete professional education,Complete professional education,8,0,4
6,Complete professional education,Complete professional education,8,0,4
7,Incomplete Secundary,Complete Secundary,7,0,0
8,Complete Secundary,Complete professional education,8,0,2
9,Incomplete technical or technological,Incomplete technical or technological,8,2,0
11,Complete technique or technology,Complete Secundary,8,2,0
13,Complete professional education,Complete professional education,8,0,4


In [7]:
df1[['EDU_BC_PARENTS','EDU_TH_PARENTS','EDU_PH_PARENTS']].describe().T

,count,mean,std,min,25%,50%,75%,max
EDU_BC_PARENTS,11491.0,7.194239,1.528393,0.0,7.0,8.0,8.0,8.0
EDU_TH_PARENTS,11491.0,0.514490,1.031039,0.0,0.0,0.0,0.0,4.0
EDU_PH_PARENTS,11491.0,1.652598,1.930002,0.0,0.0,0.0,3.0,6.0


In [8]:
df1["TOTAL_S11"] = df1[["MAT_S11", "CR_S11", "CC_S11", "BIO_S11", "ENG_S11"]].sum(axis=1)
df1["TOTAL_PRO"] = df1[["CR_PRO", "QR_PRO", "CC_PRO", "WC_PRO", "ENG_PRO"]].sum(axis=1)

In [9]:
#descriptive statistics of parental education
ds1 = df1.groupby("EDU_FATHER").agg(
    COUNT=("EDU_FATHER", "size"),
    MEAN_TOTAL_S11=("TOTAL_S11", "mean"),
    MEAN_TOTAL_PRO=("TOTAL_PRO", "mean")
).sort_values("COUNT", ascending=False)
display(ds1)

ds1 = df1.groupby("EDU_MOTHER").agg(
    COUNT=("EDU_MOTHER", "size"),
    MEAN_TOTAL_S11=("TOTAL_S11", "mean"),
    MEAN_TOTAL_PRO=("TOTAL_PRO", "mean")
).sort_values("COUNT", ascending=False)
display(ds1)

,COUNT,MEAN_TOTAL_S11,MEAN_TOTAL_PRO
EDU_FATHER,,,
Complete professional education,2995,324.770618,340.456427
Complete Secundary,2831,301.154009,303.591664
Complete technique or technology,1188,314.205387,327.112795
Incomplete Secundary,1076,296.632900,294.852230
Postgraduate education,1071,348.019608,377.178338
Complete primary,815,292.954601,288.885890
Incomplete primary,692,295.917630,290.410405
Incomplete Professional Education,425,321.684706,340.272941
Incomplete technical or technological,276,308.014493,319.329710


,COUNT,MEAN_TOTAL_S11,MEAN_TOTAL_PRO
EDU_MOTHER,,,
Complete Secundary,3020,300.421192,302.675497
Complete professional education,2985,327.519933,345.471692
Complete technique or technology,1463,313.310321,323.229665
Incomplete Secundary,1028,294.448444,290.016537
Postgraduate education,968,343.235537,372.042355
Complete primary,687,292.756914,288.585153
Incomplete Professional Education,488,320.258197,338.030738
Incomplete primary,485,293.686598,287.323711
Incomplete technical or technological,334,311.254491,319.083832


In [10]:
#descriptive statistics of parental education
def descriptive_profile(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)

    return pd.Series({
        "count": series.count(),
        "mean": series.mean(),
        "median": series.median(),
        "mode": series.mode().iloc[0],
        "min": series.min(),
        "max": series.max(),
        "range": series.max() - series.min(),
        "variance": series.var(),
        "std_dev": series.std(),
        "Q1": q1,
        "Q3": q3,
        "IQR": q3 - q1,
        "skewness": series.skew()
    })


# Father's education
father_profile = (
    df1.groupby("EDU_FATHER")[["TOTAL_S11", "TOTAL_PRO"]].apply(lambda x: x.apply(descriptive_profile).T)
)
display(father_profile.round(2))


# Mother's education
mother_profile = (
    df1.groupby("EDU_MOTHER")[["TOTAL_S11", "TOTAL_PRO"]].apply(lambda x: x.apply(descriptive_profile).T)
)

display(mother_profile.round(2))

count    mean  median  \
EDU_FATHER                                                                
Complete Secundary                    TOTAL_S11  2831.0  301.15   299.0   
                                      TOTAL_PRO  2831.0  303.59   312.0   
Complete primary                      TOTAL_S11   815.0  292.95   289.0   
                                      TOTAL_PRO   815.0  288.89   293.0   
Complete professional education       TOTAL_S11  2995.0  324.77   324.0   
                                      TOTAL_PRO  2995.0  340.46   354.0   
Complete technique or technology      TOTAL_S11  1188.0  314.21   311.0   
                                      TOTAL_PRO  1188.0  327.11   336.0   
Incomplete Professional Education     TOTAL_S11   425.0  321.68   319.0   
                                      TOTAL_PRO   425.0  340.27   352.0   
Incomplete Secundary                  TOTAL_S11  1076.0  296.63   293.5   
                                      TOTAL_PRO  1076.0  294.85   302.0   
Incomplete primary                    TOTAL_S11   692.0  295.92   292.0   
                                      TOTAL_PRO   692.0  290.41   295.5   
Incomplete technical or technological TOTAL_S11   276.0  308.01   309.0   
                                      TOTAL_PRO   276.0  319.33   325.0   
Ninguno                               TOTAL_S11   122.0  296.53   304.5   
                                      TOTAL_PRO   122.0  298.31   314.5   
Postgraduate education                TOTAL_S11  1071.0  348.02   350.0   
                                      TOTAL_PRO  1071.0  377.18   393.0   

                                                  mode    min    max  range  \
EDU_FATHER                                                                    
Complete Secundary                    TOTAL_S11  273.0  189.0  468.0  279.0   
                                      TOTAL_PRO  254.0   15.0  491.0  476.0   
Complete primary                      TOTAL_S11  288.0  189.0  439.0  250.0   
                                      TOTAL_PRO  304.0   26.0  491.0  465.0   
Complete professional education       TOTAL_S11  317.0  194.0  460.0  266.0   
                                      TOTAL_PRO  445.0   23.0  497.0  474.0   
Complete technique or technology      TOTAL_S11  285.0  209.0  478.0  269.0   
                                      TOTAL_PRO  361.0   25.0  491.0  466.0   
Incomplete Professional Education     TOTAL_S11  315.0  216.0  448.0  232.0   
                                      TOTAL_PRO  341.0   73.0  492.0  419.0   
Incomplete Secundary                  TOTAL_S11  291.0  179.0  446.0  267.0   
                                      TOTAL_PRO  262.0   21.0  487.0  466.0   
Incomplete primary                    TOTAL_S11  291.0  189.0  459.0  270.0   
                                      TOTAL_PRO  210.0   32.0  493.0  461.0   
Incomplete technical or technological TOTAL_S11  321.0  212.0  422.0  210.0   
                                      TOTAL_PRO  296.0   67.0  487.0  420.0   
Ninguno                               TOTAL_S11  275.0  194.0  413.0  219.0   
                                      TOTAL_PRO  397.0   28.0  486.0  458.0   
Postgraduate education                TOTAL_S11  370.0  214.0  475.0  261.0   
                                      TOTAL_PRO  431.0   76.0  500.0  424.0   

                                                 variance  std_dev      Q1  \
EDU_FATHER                                                                   
Complete Secundary                    TOTAL_S11   2021.26    44.96  269.00   
                                      TOTAL_PRO   9435.44    97.14  234.00   
Complete primary                      TOTAL_S11   1778.05    42.17  263.00   
                                      TOTAL_PRO   9636.34    98.16  219.00   
Complete professional education       TOTAL_S11   2320.09    48.17  290.00   
                                      TOTAL_PRO   8817.25    93.90  276.00   
Complete technique or technology    

count    mean  median  \
EDU_MOTHER                                                                
Complete Secundary                    TOTAL_S11  3020.0  300.42   298.0   
                                      TOTAL_PRO  3020.0  302.68   310.0   
Complete primary                      TOTAL_S11   687.0  292.76   288.0   
                                      TOTAL_PRO   687.0  288.59   291.0   
Complete professional education       TOTAL_S11  2985.0  327.52   326.0   
                                      TOTAL_PRO  2985.0  345.47   360.0   
Complete technique or technology      TOTAL_S11  1463.0  313.31   311.0   
                                      TOTAL_PRO  1463.0  323.23   333.0   
Incomplete Professional Education     TOTAL_S11   488.0  320.26   319.0   
                                      TOTAL_PRO   488.0  338.03   349.0   
Incomplete Secundary                  TOTAL_S11  1028.0  294.45   292.0   
                                      TOTAL_PRO  1028.0  290.02   296.5   
Incomplete primary                    TOTAL_S11   485.0  293.69   291.0   
                                      TOTAL_PRO   485.0  287.32   293.0   
Incomplete technical or technological TOTAL_S11   334.0  311.25   312.0   
                                      TOTAL_PRO   334.0  319.08   333.5   
Ninguno                               TOTAL_S11    33.0  297.91   294.0   
                                      TOTAL_PRO    33.0  269.70   294.0   
Postgraduate education                TOTAL_S11   968.0  343.24   346.5   
                                      TOTAL_PRO   968.0  372.04   392.0   

                                                  mode    min    max  range  \
EDU_MOTHER                                                                    
Complete Secundary                    TOTAL_S11  276.0  179.0  439.0  260.0   
                                      TOTAL_PRO  360.0   21.0  493.0  472.0   
Complete primary                      TOTAL_S11  263.0  193.0  439.0  246.0   
                                      TOTAL_PRO  247.0   32.0  488.0  456.0   
Complete professional education       TOTAL_S11  323.0  194.0  478.0  284.0   
                                      TOTAL_PRO  388.0   26.0  499.0  473.0   
Complete technique or technology      TOTAL_S11  303.0  200.0  461.0  261.0   
                                      TOTAL_PRO  333.0   66.0  498.0  432.0   
Incomplete Professional Education     TOTAL_S11  283.0  208.0  448.0  240.0   
                                      TOTAL_PRO  344.0   25.0  492.0  467.0   
Incomplete Secundary                  TOTAL_S11  274.0  189.0  440.0  251.0   
                                      TOTAL_PRO  234.0   15.0  482.0  467.0   
Incomplete primary                    TOTAL_S11  291.0  186.0  459.0  273.0   
                                      TOTAL_PRO  219.0   51.0  493.0  442.0   
Incomplete technical or technological TOTAL_S11  322.0  212.0  422.0  210.0   
                                      TOTAL_PRO  347.0   67.0  487.0  420.0   
Ninguno                               TOTAL_S11  272.0  210.0  387.0  177.0   
                                      TOTAL_PRO  341.0   28.0  478.0  450.0   
Postgraduate education                TOTAL_S11  356.0  206.0  475.0  269.0   
                                      TOTAL_PRO  440.0   85.0  500.0  415.0   

                                                 variance  std_dev      Q1  \
EDU_MOTHER                                                                   
Complete Secundary                    TOTAL_S11   1922.76    43.85  269.00   
                                      TOTAL_PRO   9234.65    96.10  234.00   
Complete primary                      TOTAL_S11   1865.33    43.19  262.50   
                                      TOTAL_PRO   9508.37    97.51  216.50   
Complete professional education       TOTAL_S11   2386.65    48.85  291.00   
                                      TOTAL_PRO   8683.61    93.19  282.00   
Complete technique or technology    

In [11]:
#statistical analysis
import statsmodels.formula.api as stat
#s11
mod_s11 = stat.ols("TOTAL_S11 ~ EDU_BC_PARENTS + EDU_TH_PARENTS + EDU_PH_PARENTS", data=df1).fit()
print(mod_s11.summary())
#pro
mod_pro = stat.ols("TOTAL_PRO ~ EDU_BC_PARENTS + EDU_TH_PARENTS + EDU_PH_PARENTS", data=df1).fit()
print(mod_pro.summary())

                            OLS Regression Results                            
Dep. Variable:              TOTAL_S11   R-squared:                       0.132
Model:                            OLS   Adj. R-squared:                  0.132
Method:                 Least Squares   F-statistic:                     581.3
Date:                Wed, 30 Sep 2026   Prob (F-statistic):               0.00
Time:                        06:18:22   Log-Likelihood:                -60058.
No. Observations:               11491   AIC:                         1.201e+05
Df Residuals:                   11487   BIC:                         1.202e+05
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                     coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------
Intercept        288.6823      2.128    135.

In [12]:
mod_math = stat.ols("MAT_S11 ~ EDU_BC_PARENTS + EDU_TH_PARENTS + EDU_PH_PARENTS",data=df1).fit()
mod_bio = stat.ols("BIO_S11 ~ EDU_BC_PARENTS + EDU_TH_PARENTS + EDU_PH_PARENTS",data=df1).fit()
mod_qr = stat.ols("QR_PRO ~ EDU_BC_PARENTS + EDU_TH_PARENTS + EDU_PH_PARENTS",data=df1).fit()
print(mod_math.summary())
print(mod_bio.summary())
print(mod_qr.summary())

                            OLS Regression Results                            
Dep. Variable:                MAT_S11   R-squared:                       0.089
Model:                            OLS   Adj. R-squared:                  0.089
Method:                 Least Squares   F-statistic:                     375.2
Date:                Wed, 30 Sep 2026   Prob (F-statistic):          1.63e-232
Time:                        06:18:22   Log-Likelihood:                -44234.
No. Observations:               11491   AIC:                         8.848e+04
Df Residuals:                   11487   BIC:                         8.851e+04
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                     coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------
Intercept         59.8575      0.537    111.

Which groups/stratum performs best? Are there differences between groups/stratums? Explain your analysis.

In [13]:
ap['STRATUM'].unique()

array(['Stratum 4', 'Stratum 5', 'Stratum 2', 'Stratum 6', 'Stratum 3',
       'Stratum 1', 0], dtype=object)

In [14]:
df2 = ap[~ap["STRATUM"].isin([0, "0"])].copy()

In [15]:
df2["TOTAL_S11"] = df2[["MAT_S11", "CR_S11", "CC_S11", "BIO_S11", "ENG_S11"]].sum(axis=1)
df2["TOTAL_PRO"] = df2[["CR_PRO", "QR_PRO", "CC_PRO", "WC_PRO", "ENG_PRO"]].sum(axis=1)

df2.groupby("STRATUM")[["TOTAL_S11", "TOTAL_PRO"]].mean()

,TOTAL_S11,TOTAL_PRO
STRATUM,,
Stratum 1,287.850790,275.221182
Stratum 2,299.725490,301.492182
Stratum 3,313.455624,326.014586
Stratum 4,334.267427,359.142586
Stratum 5,350.548183,378.342812
Stratum 6,362.394541,391.240695


In [16]:
#descriptive statistics of Stratum
stratum_profile = (
    df2.groupby("STRATUM")[["TOTAL_S11", "TOTAL_PRO"]]
       .apply(lambda x: x.apply(descriptive_profile).T)
)

display(stratum_profile.round(2))

count    mean  median   mode    min    max  range  \
STRATUM                                                                   
Stratum 1 TOTAL_S11  1709.0  287.85   284.0  264.0  189.0  436.0  247.0   
          TOTAL_PRO  1709.0  275.22   275.0  342.0   26.0  488.0  462.0   
Stratum 2 TOTAL_S11  4029.0  299.73   297.0  290.0  179.0  469.0  290.0   
          TOTAL_PRO  4029.0  301.49   309.0  329.0   15.0  500.0  485.0   
Stratum 3 TOTAL_S11  4045.0  313.46   311.0  308.0  183.0  475.0  292.0   
          TOTAL_PRO  4045.0  326.01   334.0  361.0   25.0  498.0  473.0   
Stratum 4 TOTAL_S11  1578.0  334.27   334.0  334.0  199.0  478.0  279.0   
          TOTAL_PRO  1578.0  359.14   374.5  437.0   43.0  499.0  456.0   
Stratum 5 TOTAL_S11   633.0  350.55   352.0  379.0  226.0  460.0  234.0   
          TOTAL_PRO   633.0  378.34   393.0  453.0   95.0  499.0  404.0   
Stratum 6 TOTAL_S11   403.0  362.39   364.0  370.0  204.0  473.0  269.0   
          TOTAL_PRO   403.0  391.24   413.0  453.0   96.0  499.0  403.0   

                     variance  std_dev     Q1     Q3    IQR  skewness  
STRATUM                                                                
Stratum 1 TOTAL_S11   1872.45    43.27  257.0  316.0   59.0      0.42  
          TOTAL_PRO   9591.04    97.93  204.0  348.0  144.0     -0.08  
Stratum 2 TOTAL_S11   1933.42    43.97  268.0  329.0   61.0      0.32  
          TOTAL_PRO   9354.64    96.72  232.0  377.0  145.0     -0.25  
Stratum 3 TOTAL_S11   2028.14    45.03  281.0  344.0   63.0      0.25  
          TOTAL_PRO   8358.37    91.42  266.0  398.0  132.0     -0.45  
Stratum 4 TOTAL_S11   2095.54    45.78  303.0  366.0   63.0     -0.02  
          TOTAL_PRO   7294.71    85.41  306.0  428.0  122.0     -0.71  
Stratum 5 TOTAL_S11   2008.78    44.82  321.0  381.0   60.0     -0.07  
          TOTAL_PRO   6383.57    79.90  335.0  442.0  107.0     -0.85  
Stratum 6 TOTAL_S11   1734.39    41.65  339.0  391.0   52.0     -0.37  
          TOTAL_PRO   6417.58    80.11  352.0  451.0   99.0     -1.11

In [17]:
stn_map={
    "Stratum 1": 1,
    "Stratum 2": 2,
    "Stratum 3": 3,
    "Stratum 4": 4,
    "Stratum 5": 5,
    "Stratum 6": 6
}
#STN for STRATUM number
df2['STN']=df2['STRATUM'].map(stn_map)

In [18]:
import statsmodels.formula.api as stat

model_s11_stn = stat.ols("TOTAL_S11 ~ STN",data=df2).fit()
model_pro_stn = stat.ols("TOTAL_PRO ~ STN", data=df2).fit()

print(model_s11_stn.summary())
print(model_pro_stn.summary())

                            OLS Regression Results                            
Dep. Variable:              TOTAL_S11   R-squared:                       0.149
Model:                            OLS   Adj. R-squared:                  0.149
Method:                 Least Squares   F-statistic:                     2171.
Date:                Wed, 30 Sep 2026   Prob (F-statistic):               0.00
Time:                        06:18:22   Log-Likelihood:                -64636.
No. Observations:               12397   AIC:                         1.293e+05
Df Residuals:                   12395   BIC:                         1.293e+05
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept    269.0991      0.996    270.241      0.0

How does each gender perform in the global assessment?

In [19]:
ap['GENDER'].unique()

array(['F', 'M'], dtype=object)

In [20]:
df3=ap
df3["TOTAL_S11"] = df3[["MAT_S11", "CR_S11", "CC_S11", "BIO_S11", "ENG_S11"]].sum(axis=1)
df3["TOTAL_PRO"] = df3[["CR_PRO", "QR_PRO", "CC_PRO", "WC_PRO", "ENG_PRO"]].sum(axis=1)
df3.groupby("GENDER")[["TOTAL_S11", "TOTAL_PRO"]].mean()

,TOTAL_S11,TOTAL_PRO
GENDER,,
F,306.507039,314.955780
M,315.011672,323.461183


In [21]:
gender_profile = (
    df3.groupby("GENDER")[["TOTAL_S11", "TOTAL_PRO"]]
       .apply(lambda x: x.apply(descriptive_profile).T)
)

display(gender_profile.round(2))

count    mean  median   mode    min    max  range  \
GENDER                                                                 
F      TOTAL_S11  5043.0  306.51   304.0  315.0  179.0  478.0  299.0   
       TOTAL_PRO  5043.0  314.96   323.0  361.0   21.0  500.0  479.0   
M      TOTAL_S11  7368.0  315.01   313.0  308.0  181.0  475.0  294.0   
       TOTAL_PRO  7368.0  323.46   334.0  342.0   15.0  499.0  484.0   

                  variance  std_dev     Q1     Q3    IQR  skewness  
GENDER                                                              
F      TOTAL_S11   2111.75    45.95  273.0  338.0   65.0      0.26  
       TOTAL_PRO   9485.84    97.40  245.0  395.0  150.0     -0.35  
M      TOTAL_S11   2441.28    49.41  279.0  350.0   71.0      0.21  
       TOTAL_PRO   9455.46    97.24  255.0  399.0  144.0     -0.43

In [22]:
#genn for gender number
genn_map={
    "F": 0,
    "M": 1
}
#STN for STRATUM number
df3['GENN']=df3['GENDER'].map(genn_map)

In [23]:
model_s11_gender = stat.ols("TOTAL_S11 ~ GENN",data=df3).fit()
model_pro_gender = stat.ols("TOTAL_PRO ~ GENN",data=df3).fit()

print(model_s11_gender.summary())
print(model_pro_gender.summary())

                            OLS Regression Results                            
Dep. Variable:              TOTAL_S11   R-squared:                       0.008
Model:                            OLS   Adj. R-squared:                  0.007
Method:                 Least Squares   F-statistic:                     93.85
Date:                Wed, 30 Sep 2026   Prob (F-statistic):           4.08e-22
Time:                        06:18:23   Log-Likelihood:                -65664.
No. Observations:               12411   AIC:                         1.313e+05
Df Residuals:                   12409   BIC:                         1.313e+05
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept    306.5070      0.676    453.132      0.0

Use your analysis to understand the results.  What did you find interesting, and can you make any recommendations or inferences?  